In [52]:
import requests

url = "https://scsanctions.un.org/consolidated"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    with open("bronze/sanctions.html", "wb") as f:
        f.write(response.content)
else:
    print(f"Failed with status code: {response.status_code}")

In [53]:
import re
import pandas as pd
from bs4 import BeautifulSoup

# Load HTML content
with open('bronze/sanctions.html', 'r', encoding='utf-8') as file:
    soup = BeautifulSoup(file, 'html.parser')

# Get full text from HTML
full_text = soup.get_text()

# Step 1: Find all occurrences of the section header
matches = list(re.finditer(r'B\.\s*Entities and other groups', full_text, re.IGNORECASE))

if len(matches) < 2:
    raise ValueError("Could not find the second 'B. Entities and other groups' section.")

# Step 2: Start from the second match
start_index = matches[1].end()

# Step 3: Extract everything from there until the next top-level section or EOF
end_match = re.search(r'^[A-Z]\.\s+[^\n]+', full_text[start_index:], re.MULTILINE)
end_index = end_match.start() + start_index if end_match else len(full_text)
section_text = full_text[start_index:end_index].strip()

# Step 4: Define ID patterns
id_patterns = [
    r'TAe\.\d{3}',
    r'QDe\.\d{3}',
    r'YEe\.\d{3}',
    r'KPe\.\d{3}',
    r'CFe\.\d{3}',
    r'CDe\.\d{3}',
    r'LYe\.\d{3}',
    r'SOe\.\d{3}',
    r'IQe\.\d{3}'
]
combined_pattern = '|'.join(id_patterns)

# Step 5: Use finditer to get real entries only (ID followed by "Name:")
entry_starts = list(re.finditer(rf'({combined_pattern})\s+Name:', section_text))

# Slice entries safely using start/end indices
entries = []
for i in range(len(entry_starts)):
    start = entry_starts[i].start()
    end = entry_starts[i + 1].start() if i + 1 < len(entry_starts) else len(section_text)
    entries.append(section_text[start:end])

# Step 6: Extract structured data
data = []

for entry in entries:
    id_match = re.search(combined_pattern, entry)
    name_match = re.search(r'Name:\s*(.*?)\s*A\.k\.a\.:', entry, re.DOTALL)
    aka_match = re.search(r'A\.k\.a\.: (.*?)(?:F\.k\.a\.|Address:|Listed on:)', entry, re.DOTALL)
    fka_match = re.search(r'F\.k\.a\.: (.*?)(?:Address:|Listed on:)', entry, re.DOTALL)
    address_match = re.search(r'Address:\s*(.*?)(?:Listed on:|Other information:)', entry, re.DOTALL)
    listed_on_match = re.search(r'Listed on:\s*([0-9]{1,2} [A-Za-z]+(?:\.?) \d{4})', entry)
    other_info_match = re.search(r'Other information:\s*(.*)', entry, re.DOTALL)

    row = {
        "id": id_match.group(0).strip() if id_match else "",
        "name": name_match.group(1).strip() if name_match else "",
        "other_names": aka_match.group(1).strip() if aka_match else "",
        "previous_names": fka_match.group(1).strip() if fka_match else "",
        "address": address_match.group(1).strip() if address_match else "",
        "listed_on": listed_on_match.group(1).strip() if listed_on_match else "",
        "other_information": other_info_match.group(1).strip() if other_info_match else "",
    }

    data.append(row)

# Step 7: Create DataFrame and export
df = pd.DataFrame(data)
df.to_csv('silver/sanctions_entities.csv', sep='|', index=False)

print(df.head())

        id                                               name  \
0  QDe.069                     AFGHAN SUPPORT COMMITTEE (ASC)   
1  QDe.121                      AL-AKHTAR TRUST INTERNATIONAL   
2  QDe.109  AL-HARAMAIN & AL MASJED AL-AQSA CHARITY FOUNDA...   
3  QDe.113                       AL-HARAMAIN: ETHIOPIA BRANCH   
4  QDe.116      AL-HARAMAIN FOUNDATION (UNION OF THE COMOROS)   

                                         other_names previous_names  \
0  a) Lajnat ul Masa Eidatul Afghania b) Jamiat A...             na   
1  a) Al Akhtar Trust b) Al-Akhtar Medical Centre...             na   
2  a) Al Haramain Al Masjed Al Aqsa b) Al Haramay...             na   
3                                                 na             na   
4                                                 na             na   

                                             address     listed_on  \
0  a) Headquarters – G.T. Road (probably Grand Tr...  11 Jan. 2002   
1  a) ST-1/A, Gulsahn-e-Iqbal, Block 2, Ka